# Ch 29. Tensor and Pipeline Parallelism — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: Implement column and row parallelism and show that their results equal the original.

### Solution

Column split partitions $W=[W_1,\ldots,W_k]$ and concatenates $[XW_1,\ldots,XW_k]=XW$. Row split partitions matching input/weight rows and all-reduces $\sum_iX_iW_i=XW$.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
rng=np.random.default_rng(29); x=rng.normal(size=(3,8)); W=rng.normal(size=(8,12)); ref=x@W
column=np.concatenate([x@w for w in np.split(W,4,axis=1)],axis=1)
row=sum(xx@ww for xx,ww in zip(np.split(x,4,axis=1),np.split(W,4,axis=0)))
assert np.allclose(ref,column) and np.allclose(ref,row); ref.shape


## Problem 2

**Problem**: Calculate the pipeline bubble fraction for p=2, 4, 8 and m=4, 16, 64.

### Solution

For GPipe with $p$ equal-time stages and $m$ microbatches, there are $m+p-1$ slots and $p-1$ empty slots, so the bubble fraction is $(p-1)/(m+p-1)$.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
bubble={(p,m):(p-1)/(m+p-1) for p in [2,4,8] for m in [4,16,64]}
assert bubble[8,4]>bubble[2,64]; bubble


## Problem 3

**Problem**: Simulate tensor parallelism that splits attention heads across 4 GPUs.

### Solution

Attention heads are independent through softmax, so splitting the head axis four ways, computing each shard, and concatenating in original order equals the single-device result.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
rng=np.random.default_rng(3); heads=rng.normal(size=(2,8,5,4)); shards=np.split(heads,4,axis=1); gathered=np.concatenate(shards,axis=1)
assert np.array_equal(heads,gathered) and all(s.shape[1]==2 for s in shards); gathered.shape


## Problem 4

**Problem**: Explain why 1024 GPUs can be configured as DP=8, TP=8, PP=16 in 3D parallelism.

### Solution

Because $8\times8\times16=1024$: TP fits each layer by 8-way sharding, PP divides layers into 16 stages, and DP=8 replicates the whole combination for throughput.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
DP,TP,PP=8,8,16
assert DP*TP*PP==1024
{'replicas':DP,'intra_layer_shards':TP,'pipeline_stages':PP,'total':DP*TP*PP}


## Problem 5

**Problem**: Explain how 1F1B scheduling saves memory.

### Solution

Peak live activations under 1F1B are not a constant two; they depend on pipeline depth, stage index, microbatch count, and warmup. The reduced simulator allocates an activation at each stage-local forward and releases it at the matching backward. Its simulated peak is checked against an independently computed maximum overlap of activation lifetime intervals.

The dependency-free code computes per-stage peaks and the event trace.


In [ ]:
from collections import deque

def one_f_one_b_stage(depth, stage, microbatches):
    warmup = min(depth - stage - 1, microbatches)
    forward_next, backward_next = 0, 0
    live, intervals, events = {}, {}, []

    def forward(mb):
        live[mb] = len(events)
        events.append(('F', mb, len(live)))

    def backward(mb):
        start = live.pop(mb)
        events.append(('B', mb, len(live)))
        intervals[mb] = (start, len(events) - 1)

    for _ in range(warmup):
        forward(forward_next); forward_next += 1
    while forward_next < microbatches:
        forward(forward_next); forward_next += 1
        backward(backward_next); backward_next += 1
    while backward_next < microbatches:
        backward(backward_next); backward_next += 1

    simulated_peak = max(count for _, _, count in events)
    reference_peak = max(
        sum(start <= tick < end for start, end in intervals.values())
        for tick in range(len(events))
    )
    assert simulated_peak == reference_peak and not live
    return {'stage': stage, 'warmup': warmup, 'peak_live': simulated_peak,
            'events': events, 'intervals': intervals}

depth, microbatches = 4, 8
stage_results = [one_f_one_b_stage(depth, stage, microbatches) for stage in range(depth)]
peaks = [row['peak_live'] for row in stage_results]
assert peaks == [4, 3, 2, 1]
{'depth': depth, 'microbatches': microbatches, 'per_stage_peak_live': peaks}
